## Generate Sample Dataset

In [ ]:
import pandas as pd
import numpy as np


df = pd.read_parquet("/home/sakib/Documents/promptmark/datasets/human_eval_164.parquet")
df.head()

In [ ]:
# generate 5 rand number with reproducibility
np.random.seed(42)
rand_indices = np.random.randint(0, 164, size=150)
# the issue was they were not unique

In [ ]:
df_selected = df.iloc[rand_indices]
df_selected.to_csv('sample_humaneval.csv', index=False)

In [ ]:
import os 

files = os.listdir("/home/sakib/Documents/promptmark/output/gemini_exp4_during_gen_v1_161_humaneval")
print(len(files))
available_files = []

for file in files:
    idx = int(file.split("_")[-1].split(".")[0])
    if idx in rand_indices:
        available_files.append(idx)

print("Available: ", len(available_files))

In [ ]:
org_code = []
wm_code = []

for idx in available_files:

    with open(f"/home/sakib/Documents/promptmark/output/gemini_exp4_during_gen_v1_161_humaneval/{idx}.py", "r") as f:
        wm_code.append(f.read())
        org_code.append(df.iloc[idx]['canonical_solution'])

In [ ]:
df_new = pd.DataFrame({
    "pid": available_files,
    "original_code": org_code,
    "watermarked_code": wm_code
})

In [ ]:
print(len(org_code))
print(len(wm_code))
print(df_new.shape)

In [ ]:
df_new.head()

In [ ]:
df_new[df_new['pid'] == 92 ]['original_code']

In [ ]:
df_new.to_csv("/home/sakib/Documents/promptmark/scripts/robustness/data/csvs/exp_4_data.csv/home/sakib/Documents/promptmark/scripts/robustness/data/csvs/exp_4_data.csv", index=False)

## Parapharase using Llama-8b

In [ ]:
import pandas as pd

In [ ]:
df_test = pd.read_csv("/home/sakib/Documents/promptmark/scripts/robustness/data/csvs/exp_4_data.csv")
df_test.shape

In [ ]:
import requests
import json
import re

# Ollama API endpoint
url = "http://localhost:11434/api/generate"

def extract_python_code(text):
    # Try to extract code from markdown code blocks
    code_blocks = re.findall(r'```python\n(.*?)\n```', text, re.DOTALL)
    if code_blocks:
        return code_blocks[0]
    
    # Try to extract code from generic code blocks
    code_blocks = re.findall(r'```\n(.*?)\n```', text, re.DOTALL)
    if code_blocks:
        return code_blocks[0]
    
    # If no code blocks found, return the original text
    return text

# Function to generate inference using Llama-8B
def generate_inference(prompt):
    payload = {
        "model": "qwen2.5-coder:3b",
        "prompt": prompt,
        "stream": False
    }
    
    response = requests.post(url, json=payload)
    if response.status_code == 200:
        return response.json()["response"]
    else:
        return f"Error: {response.status_code}"

# Generate paraphrased code for each row
paraphrased_code = []
for idx, row in df_test.iterrows():
    prompt = f"Paraphrase the following Python code:\n\n{row['watermarked_code']}"
    result = generate_inference(prompt)
    python_code = extract_python_code(result)
    paraphrased_code.append(python_code)
    print(f"Processed row {idx}")

df_test['paraphrased_code_qwen'] = paraphrased_code

In [ ]:
df_test.to_csv('/home/sakib/Documents/promptmark/scripts/robustness/data/csvs/humaneval_paraphrased_qwen.csv', index=False)

In [ ]:
import requests

# Ollama API endpoint
url = "http://localhost:11434/api/generate"

# Function to generate inference using Llama-8B
def generate_inference(prompt):
    payload = {
        "model": "llama3.1:8b",
        "prompt": prompt,
        "stream": False
    }
    
    response = requests.post(url, json=payload)
    if response.status_code == 200:
        return response.json()["response"]
    else:
        return f"Error: {response.status_code}"

# Generate paraphrased code for each row
paraphrased_code = []
for idx, row in df_test.iterrows():
    prompt = f"Paraphrase the following Python code:\n\n{row['watermarked_code']}"
    result = generate_inference(prompt)
    python_code = extract_python_code(result)
    paraphrased_code.append(python_code)
    print(f"Processed row {idx}")

df_test['paraphrased_code_llama'] = paraphrased_code

In [ ]:
df_test.to_csv('/home/sakib/Documents/promptmark/scripts/robustness/data/csvs/humaneval_paraphrased_combined.csv', index=False)

In [ ]:
import pandas as pd
import os
df = pd.read_parquet('data/human_eval.parquet')
df.head()

# int(df.iloc[0]['task_id'].split('/')[1])

In [ ]:
import textwrap

# save the original code separate folder
os.makedirs('data/original_codes', exist_ok=True)
for idx, row in df.iterrows():
    pid = int(row['task_id'].split('/')[1])
    
    prompt = row['prompt']
    code = row['canonical_solution']
    complete_code = prompt + "\n" + code
    complete_code = textwrap.dedent(complete_code)

    open(f'data/original_codes/{pid}.py', 'w').write(complete_code)

## Prepare Folder

In [ ]:
df_root = pd.read_csv('/home/sakib/Documents/promptmark/scripts/robustness/data/csvs/exp_4_data.csv')
df_test = pd.read_csv('/home/sakib/Documents/promptmark/scripts/robustness/data/csvs/humaneval_paraphrased_combined.csv')

df_test['paraphrased_codes_static'] = df_root['perturbed_code_static']
df_test.head()


In [ ]:
os.makedirs('/home/sakib/Documents/promptmark/scripts/robustness/data/paraphrased_codes_llama', exist_ok=True)
os.makedirs('/home/sakib/Documents/promptmark/scripts/robustness/data/paraphrased_codes_qwen', exist_ok=True)

In [ ]:
os.makedirs('/home/sakib/Documents/promptmark/scripts/robustness/data/paraphrased_codes_static', exist_ok=True)

In [ ]:
for idx, row in df_test.iterrows():

    lama_code = row['paraphrased_code_llama']
    qwen_code = row['paraphrased_code_qwen']
    static_code = row['paraphrased_codes_static']

    open(f'/home/sakib/Documents/promptmark/scripts/robustness/data/paraphrased_codes_llama/{row["pid"]}.py', 'w').write(lama_code)
    open(f'/home/sakib/Documents/promptmark/scripts/robustness/data/paraphrased_codes_qwen/{row["pid"]}.py', 'w').write(qwen_code)
    open(f'/home/sakib/Documents/promptmark/scripts/robustness/data/paraphrased_codes_static/{row["pid"]}.py', 'w').write(static_code)

## Prepare Raw HumanEval Data


In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('data/csvs/sample_humaneval.csv')
df.head()

In [ ]:
df_code = df['prompt'].str.cat(df['canonical_solution'], sep="\n")
df_code = df_code.str.lstrip('\n')
df['complete_code'] = df_code

# remove the leading \n
open('temp.py', 'w').write(df_code[10])

In [ ]:
import os
os.makedirs('data/humaneval', exist_ok=True)

for idx, row in df.iterrows():
    complete_code = row['complete_code']
    open(f'data/humaneval/{idx}.py', 'w').write(complete_code)